# Basic figures for 311 service requests vs EMS heat calls

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

import os
import shii
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import datetime as dt

APP_TOKEN = os.getenv('NYC_OPEN_DATA_APP_TOKEN')

## 1. Prep data

In [ ]:

all_311_df = shii.prepare_all_311_requests(APP_TOKEN, output_path='./311_calls.gpkg')
heat_inc_df = shii.prepare_ems_calls(APP_TOKEN, output_path='./ems_calls.csv')

In [ ]:
#  Separate types of vent complaints
# all_311_df.loc[all_311_df['request_type']=='ventilation'].groupby('agency_name').size()
# all_311_df.loc[(all_311_df['request_type']=='ventilation') & (all_311_df['agency_name']=='Department of Health and Mental Hygiene'), 'request_type'] = 'vent_doh'
# all_311_df.loc[(all_311_df['request_type']=='ventilation') & (all_311_df['agency_name']=='Department of Housing Preservation and Development'), 'request_type'] = 'vent_house'

In [ ]:
full_df = shii.preprocess_merge_df(heat_inc_df, all_311_df, summer_only=False)

In [ ]:
full_df['hydrant_all'] = full_df['hydrant'] + full_df['fhe']

In [ ]:
# Normalize variables by population
# for col in ['ac','hydrant','power','ventilation','fhe', 'heat_ems_count']:
for col in ['ac','hydrant','hydrant_all','power','ventilation', 'fhe', 'heat_ems_count']:
    full_df[col+'_norm'] = 100000*full_df[col]/full_df['population']

In [ ]:
date_grouped = full_df.groupby('date').sum()

In [ ]:
full_df['heat_ems_count'].sum()

In [ ]:
# After 2022
post_2022_df = full_df.loc[full_df.index.get_level_values(1).year >= 2022]
cdta_grouped = post_2022_df.groupby('cdta').mean()
cdta_grouped_sum = post_2022_df.groupby('cdta').sum()
# Replace sums with mean values in cases where means don't make sense
per_cdta_vars = ['HVI_RANK', 'SURFACE_TEMP', 'MEDIAN_INCOME',
                 'GREENSPACE', 'PCT_HOUSEHOLDS_AC','PCT_BLACK_POP',
                 'population',]
cdta_grouped_sum[per_cdta_vars] = cdta_grouped[per_cdta_vars]


### Time series

In [ ]:
# Fig 2: Time series comparison
fig, axs = plt.subplots(2, 3, figsize=(12, 8))
ax_flat = axs.flatten()
start_date = '2024-05-01'
end_date = '2024-09-30'
# Heat counts
date_grouped.loc[pd.date_range(start_date, end_date)]['heat_ems_count_norm'].plot(
    xlim=(start_date, end_date),
    ax=axs[0, 0],
    xlabel='',
    ylabel='EMS Calls per 100k',
    ylim=(-4, 55),
    title='Heat Illness EMS Calls',
    lw=2
    )

# Temp
date_grouped['tmax'].plot(
    xlim=(start_date, end_date),
    ax=axs[1, 0],
    color='darkred',
    ylabel='Temperature (C)',
    xlabel='',
    title='Max Temp',
    lw=2
    )

# 311 calls
ax_list = [axs[0, 1], axs[0,2], axs[1,1], axs[1,2]]
color_list = ['#D62828', '#FCBF49', '#0077B6', '#2A9D8F']
title_list = ['311 Hydrant', '311 Ventilation', '311 Power', '311 AC']

for i, col in enumerate(['hydrant_all_norm','ventilation_norm','ac_norm','power_norm']):
    date_grouped.loc[pd.date_range(start_date, end_date)][col].plot(
        # xlim=(start_date, end_date),
        ax=ax_list[i],
        color = color_list[i],
        xlabel='',
        ylabel='Requests per 100k people',
        title=title_list[i],
        lw=2
        )
    

heat_dates = ['2024-06-21', '2024-07-06','2024-07-16','2024-08-02','2024-08-28']
for ax in axs.flatten():
    for hd in heat_dates:
        line = ax.axvline(hd, alpha=0.5, color='black', linestyle='--', lw=0.9)

axs[0,0].legend([line], ['Heat Illness Local Maxima'])
plt.tight_layout()

### Scatter plots

In [ ]:
# Fig 3: Per CDTA comparisons with heat ems counts
fig, axs = plt.subplots(3,3, figsize=(12, 8))
color_list = ['#D62828', '#FCBF49', '#0077B6', '#2A9D8F']
cdta_grouped_sum.plot.scatter(
    x='HVI_RANK',
    y='heat_ems_count',
    ax=axs[0, 0],
    color='darkorange',
    title='HVI Score',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='HVI Score (1-5)',
    )
cdta_grouped_sum.plot.scatter(
    x='MEDIAN_INCOME',
    y='heat_ems_count',
    ax=axs[1, 0],
    color='green',
    title='Median Income',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='Median Income ($/year)',
    )
cdta_grouped_sum.plot.scatter(
    x='GREENSPACE',
    y='heat_ems_count',
    ax=axs[2,0],
    color='darkgreen',
    title='Greenspace',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='Greenspace (%)',
    )
cdta_grouped_sum.plot.scatter(
    x='SURFACE_TEMP',
    y='heat_ems_count',
    ax=axs[0, 1],
    color='purple',
    title='Surface Temp',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='Surface Temp (F)',
    )
cdta_grouped_sum.plot.scatter(
    x='PCT_BLACK_POP',
    y='heat_ems_count',
    ax=axs[1, 1],
    color='darkblue',
    title='Black population %',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='% Black population'
    )
cdta_grouped_sum.plot.scatter(
    x='PCT_HOUSEHOLDS_AC',
    y='heat_ems_count',
    ax=axs[2, 1],
    color=color_list[3],
    title='Household AC %',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='% Households with AC'
    )
cdta_grouped_sum.plot.scatter(
    x='hydrant_all',
    y='heat_ems_count',
    ax=axs[0, 2],
    color=color_list[0],
    title='311 Hydrant',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='# Rqests',
    )
cdta_grouped_sum.plot.scatter(
    x='ventilation',
    y='heat_ems_count',
    ax=axs[1, 2],
    color=color_list[1],
    title='311 Ventilation',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='# Requests'
    )
cdta_grouped_sum.plot.scatter(
    x='power',
    y='heat_ems_count',
    ax=axs[2, 2],
    color=color_list[2],
    title='311 Power',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='# Requests',
    )
fig.tight_layout()

### Maps

In [ ]:
cdta_map = shii.download_community_districts().rename(columns={'boro_cd':'cdta'}).set_index('cdta')[['geometry']]

In [ ]:
import matplotlib

In [ ]:
# Fig 1: Heat EMS times series and map
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
date_grouped['heat_ems_count'].plot(
    ax=axs[0],
    xlabel='',
    ylabel='EMS Calls per day',
    title='Heat Illness EMS Calls per day',
    color='darkred'
    )


cdta_map.join(cdta_grouped_sum).plot(
    column='heat_ems_count_norm',
    legend=True,
    ax=axs[1],
    norm=matplotlib.colors.LogNorm(vmin=cdta_grouped_sum.heat_ems_count_norm.min(), vmax=cdta_grouped_sum.heat_ems_count_norm.max())
)
axs[1].set_title('Total Heat Illness EMS Calls, 2010-2025, Per 100k (Log Scale)')
axs[1].set_xlabel('Lon')
axs[1].set_ylabel('Lat')
fig.show()

# Real-time HVI

In [ ]:
full_df.columns

In [ ]:
# Get last 3 days of EMS calls and 311 requests
rolling_days = 3
rolling_columns = ['heat_ems_count','heat_ems_count_norm','ac','ac_norm','fhe','fhe_norm',
                   'hydrant','hydrant_norm','hydrant_all','hydrant_all_norm','power','power_norm',
                   'ventilation','ventilation_norm']
heat_rolling = shii.compute_rolling(full_df, rolling_columns,window=dt.timedelta(days=rolling_days))
heat_rolling = heat_rolling.reset_index().set_index(['cdta','date'])
roll_df = full_df.join(heat_rolling, rsuffix='_last3').reset_index()

In [ ]:
date_grouped.rolling(3,1).sum()['heat_ems_count_norm'].plot(xlim=['2024-06-01','2024-09-01'])

In [ ]:

roll_df.loc[roll_df.index.get_level_values(1).month.isin([6, 7,8]),
            ['ventilation_norm_last3', 'ac_norm_last3','power_norm_last3','heat_ems_count_norm_last3','hydrant_all_norm_last3']].quantile(0.8)

In [ ]:
# Add flag thresholds
def compute_shii(df):
    threshold_dict = {
        'hydrant_all_norm_last3': 8.6,
        'ventilation_norm_last3': .8,
        'ac_norm_last3':0.0,
        'heat_ems_count_norm_last3':0.5,
        'power_norm_last3': 1.0
    }
    shii_cols = []
    for crit, thresh in threshold_dict.items():
        shii_colname=f'shii_{crit}'
        df[shii_colname] = df[crit]>thresh
        shii_cols.append(shii_colname)
    
    df['shii_total'] = df[shii_cols].sum(axis=1)

    return df[['shii_total'] + shii_cols]
    

In [ ]:
shii_df = cdta_map.join(compute_shii(roll_df), how='right')

In [ ]:
cdta_grouped['HVI_RANK_ROUND'] = cdta_grouped['HVI_RANK'].round()

In [ ]:

fig, axs = plt.subplots(1, 3, figsize=(15,3))
cdta_map.join(cdta_grouped).plot(
    column='HVI_RANK_ROUND', legend=True, vmin=1, vmax=5, ax = axs[0])
axs[0].set_title('Static HVI Rank')
shii_df.reset_index().set_index(['date','cdta']).loc['2024-06-10'].plot(
    column='shii_total', legend=True, vmax=5, ax = axs[1])
axs[1].set_title('June 10, 2024')
shii_df.reset_index().set_index(['date','cdta']).loc['2024-06-21'].plot(
    column='shii_total', legend=True, vmax=5, ax = axs[2])
axs[2].set_title('June 21, 2024')
